# Exp5 LME feature decomposition

This notebook builds a clean long-format response table for Exp5 thalamic neural response amplitudes and fits editable linear mixed-effects models. The response is the same per-neuron, per-stimulus z-score AUC amplitude used in `Exp5_map_positions_several.ipynb`, computed through the shared `src.multifish_analysis.build_zscore_response_matrices_all_fish` helper. Models account for repeated stimulus measurements from the same neuron and fish by using `fish_neuron_id` as the grouping variable and `fish_id` as an optional variance component.

Edit only the stimulus metadata and `model_specs` blocks to change the scientific feature definitions or formulas.

## Setup

In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import sys
import importlib

repo_root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src").is_dir())
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

import src.analysis_tools as at
import src.data_loading as exio
import src.lme_feature_decomposition as lme
import src.multifish_analysis as mfa
import src.plotting as plott

importlib.reload(at)
importlib.reload(exio)
importlib.reload(lme)
importlib.reload(mfa)
importlib.reload(plott)

## Loading And Response Matrices

In [ ]:
# Edit paths and fish list here.
experiment_name = "Exp_5_map_positions"
main_path = Path(r"D:\Alejandro\Data\OneDrive - Université de Lausanne\Lab\Data\2p")
analysis_path = Path(r"D:\Alejandro\Data\OneDrive - Université de Lausanne\Lab\Analysis")
stimuli_main_path = analysis_path

fps_2p = 2.0
selected_blocks = [f"B{n}" for n in range(1, 5)]
t_pre_s = 5.0
t_post_s = 25.0
motion_onset_s = 8.0
tau_s = 6.0
motion_duration_key = "motion_sec"

fish_ids = [
    "L684_f01",
    "L684_f07",
    "L733_f01",
    "L733_f06",
    "L733_f07",
]

selected_stimuli = [
    "LeBcontrol", "RiBcontrol",
    "LeB1", "RiB1", "LeB2", "RiB2", "LeB3", "RiB3", "LeB4", "RiB4",
    "FlickL", "FlickR",
    "RockL", "RockR",
]

In [ ]:
def load_and_align_zscore_for_lme(
    fish_id,
    experiment_name,
    main_path,
    stimuli_main_path,
    fps_2p,
    selected_blocks,
    t_pre_s,
    t_post_s,
):
    results = exio.load_2p_experiment(
        fish_id=fish_id,
        experiment_name=experiment_name,
        main_path=main_path,
        stimuli_main_path=stimuli_main_path,
        fps_2p=fps_2p,
        selected_blocks=selected_blocks,
    )
    aligned = at.build_trial_aligned_traces(
        dfof=results["z_traces"],
        stimuli_trace_60=results["stimuli_trace_60"],
        fps_2p=fps_2p,
        t_pre_s=t_pre_s,
        t_post_s=t_post_s,
        stimuli_id_map=results["stimuli_id_map"],
        verbose=True,
    )
    return {
        "trial_aligned_traces_z_core": aligned["trial_aligned_traces"],
        "stimuli_ids": aligned["stimuli_ids"],
        "stimuli_names": aligned["stimuli_names"],
        "win_length": aligned["win_length"],
        "stimuli_trace": aligned["stimuli_trace"],
        "onsets_by_id": aligned["onsets_by_id"],
        "stimuli_durations": results["stimuli_durations"],
        "kept_neuron_indices": results["kept_neuron_indices"],
        "stimuli_id_map": results["stimuli_id_map"],
        "meta": {
            "paths": results["paths"],
            "plane_ids": results["plane_ids"],
        },
    }


all_fish_data = {}
for fish_id in fish_ids:
    print(f"\n=== Loading {fish_id} ===")
    all_fish_data[fish_id] = load_and_align_zscore_for_lme(
        fish_id=fish_id,
        experiment_name=experiment_name,
        main_path=main_path,
        stimuli_main_path=stimuli_main_path,
        fps_2p=fps_2p,
        selected_blocks=selected_blocks,
        t_pre_s=t_pre_s,
        t_post_s=t_post_s,
    )

reference_fish_id = fish_ids[0]
reference_fish = all_fish_data[reference_fish_id]
print("\nReference stimulus ID map:")
print(reference_fish["stimuli_id_map"])

In [ ]:
selection = at.resolve_selected_stimuli(
    selected_stimuli,
    stimuli_id_map=reference_fish["stimuli_id_map"],
    available_stimuli=reference_fish["trial_aligned_traces_z_core"].keys(),
)
selected_stimulus_ids = selection["stimulus_ids"]
selected_stimulus_labels = selection["stimulus_labels"]

response_window_rows = []
for fish_id in fish_ids:
    fish = all_fish_data[fish_id]
    trial_aligned_z = fish["trial_aligned_traces_z_core"]
    for stim_id, stim_label in zip(selected_stimulus_ids, selected_stimulus_labels):
        stim_key = stim_id if stim_id in trial_aligned_z else str(stim_id)
        arr = trial_aligned_z[stim_key]
        window = at.compute_response_window_frames(
            n_time=arr.shape[1],
            fps_2p=fps_2p,
            t_pre_s=t_pre_s,
            motion_onset_s=motion_onset_s,
            stimulus=stim_id,
            stimuli_durations=fish["stimuli_durations"],
            stimuli_id_map=fish["stimuli_id_map"],
            tau_s=tau_s,
            motion_duration_key=motion_duration_key,
        )
        response_window_rows.append(
            {
                "fish_id": fish_id,
                "stimulus": stim_label,
                "stimulus_id": stim_id,
                "n_time": arr.shape[1],
                "start_frame": window["start_frame"],
                "stop_frame": window["stop_frame"],
                "n_response_frames": window["n_frames"],
                "start_s": window["start_s"],
                "end_s": window["end_s"],
                "requested_end_s": window["requested_end_s"],
                "motion_duration_s": window["motion_duration_s"],
            }
        )

response_window_validation = pd.DataFrame(response_window_rows)
print("Selected stimulus IDs:", selected_stimulus_ids)
print("Selected stimulus labels:", selected_stimulus_labels)
display(response_window_validation.head())

In [ ]:
response_outputs = mfa.build_zscore_response_matrices_all_fish(
    all_fish_data=all_fish_data,
    fish_ids=fish_ids,
    selected_stimuli=selected_stimuli,
    fps_2p=fps_2p,
    t_pre_s=t_pre_s,
    motion_onset_s=motion_onset_s,
    tau_s=tau_s,
    motion_duration_key=motion_duration_key,
)

response_matrices_by_fish = response_outputs["response_matrices"]
pooled_response_matrix = response_outputs["pooled_response_matrix"]
response_row_metadata = response_outputs["row_metadata"]

expected_neurons_by_fish = {
    fish_id: len(all_fish_data[fish_id]["kept_neuron_indices"])
    for fish_id in fish_ids
}
response_matrix_shape_summary = pd.DataFrame(
    [
        {
            "fish_id": fish_id,
            "response_rows": response_matrices_by_fish[fish_id].shape[0],
            "kept_neurons": expected_neurons_by_fish[fish_id],
            "response_columns": response_matrices_by_fish[fish_id].shape[1],
        }
        for fish_id in fish_ids
    ]
)
if not np.all(response_matrix_shape_summary["response_rows"] == response_matrix_shape_summary["kept_neurons"]):
    raise ValueError("Response matrix rows do not match kept_neuron_indices for every fish.")

print("Pooled response matrix:", pooled_response_matrix.shape)
display(response_matrix_shape_summary)
display(pooled_response_matrix.head())

## Stimulus Metadata

In [ ]:
# Edit this table by eye before fitting if a category should be changed.
stimulus_metadata = pd.DataFrame(
    [
        {"stimulus_name": "LeBcontrol", "hemifield": "left", "stimulus_class": "full_bout", "position_id": "full", "motion_level": "extended", "is_bout_like": 1, "is_full_bout": 1, "is_rocking": 0, "is_flicker": 0},
        {"stimulus_name": "RiBcontrol", "hemifield": "right", "stimulus_class": "full_bout", "position_id": "full", "motion_level": "extended", "is_bout_like": 1, "is_full_bout": 1, "is_rocking": 0, "is_flicker": 0},
        {"stimulus_name": "LeB1", "hemifield": "left", "stimulus_class": "segment", "position_id": "B1", "motion_level": "minimal", "is_bout_like": 1, "is_full_bout": 0, "is_rocking": 0, "is_flicker": 0},
        {"stimulus_name": "RiB1", "hemifield": "right", "stimulus_class": "segment", "position_id": "B1", "motion_level": "minimal", "is_bout_like": 1, "is_full_bout": 0, "is_rocking": 0, "is_flicker": 0},
        {"stimulus_name": "LeB2", "hemifield": "left", "stimulus_class": "segment", "position_id": "B2", "motion_level": "minimal", "is_bout_like": 1, "is_full_bout": 0, "is_rocking": 0, "is_flicker": 0},
        {"stimulus_name": "RiB2", "hemifield": "right", "stimulus_class": "segment", "position_id": "B2", "motion_level": "minimal", "is_bout_like": 1, "is_full_bout": 0, "is_rocking": 0, "is_flicker": 0},
        {"stimulus_name": "LeB3", "hemifield": "left", "stimulus_class": "segment", "position_id": "B3", "motion_level": "minimal", "is_bout_like": 1, "is_full_bout": 0, "is_rocking": 0, "is_flicker": 0},
        {"stimulus_name": "RiB3", "hemifield": "right", "stimulus_class": "segment", "position_id": "B3", "motion_level": "minimal", "is_bout_like": 1, "is_full_bout": 0, "is_rocking": 0, "is_flicker": 0},
        {"stimulus_name": "LeB4", "hemifield": "left", "stimulus_class": "segment", "position_id": "B4", "motion_level": "minimal", "is_bout_like": 1, "is_full_bout": 0, "is_rocking": 0, "is_flicker": 0},
        {"stimulus_name": "RiB4", "hemifield": "right", "stimulus_class": "segment", "position_id": "B4", "motion_level": "minimal", "is_bout_like": 1, "is_full_bout": 0, "is_rocking": 0, "is_flicker": 0},
        {"stimulus_name": "FlickL", "hemifield": "left", "stimulus_class": "flicker", "position_id": "fixed", "motion_level": "none", "is_bout_like": 0, "is_full_bout": 0, "is_rocking": 0, "is_flicker": 1},
        {"stimulus_name": "FlickR", "hemifield": "right", "stimulus_class": "flicker", "position_id": "fixed", "motion_level": "none", "is_bout_like": 0, "is_full_bout": 0, "is_rocking": 0, "is_flicker": 1},
        {"stimulus_name": "RockL", "hemifield": "left", "stimulus_class": "rocking", "position_id": "fixed", "motion_level": "minimal", "is_bout_like": 0, "is_full_bout": 0, "is_rocking": 1, "is_flicker": 0},
        {"stimulus_name": "RockR", "hemifield": "right", "stimulus_class": "rocking", "position_id": "fixed", "motion_level": "minimal", "is_bout_like": 0, "is_full_bout": 0, "is_rocking": 1, "is_flicker": 0},
    ]
)

display(stimulus_metadata)

## Build Long-Format Table

In [ ]:
response_table = lme.build_lme_response_table(
    response_matrices_by_fish=response_matrices_by_fish,
    stimulus_metadata=stimulus_metadata,
)

category_orders = {
    "hemifield": ["left", "right"],
    "stimulus_class": ["full_bout", "segment", "rocking", "flicker"],
    "position_id": ["B1", "B2", "B3", "B4", "full", "fixed"],
    "motion_level": ["none", "minimal", "extended"],
}
for column, order in category_orders.items():
    response_table[column] = pd.Categorical(response_table[column], categories=order, ordered=False)

print("Response table shape:", response_table.shape)
display(response_table.head())

## Validation

In [ ]:
validation_summaries = lme.validate_lme_response_table(
    response_table,
    stimulus_metadata=stimulus_metadata,
    expected_neurons_by_fish=expected_neurons_by_fish,
    expected_stimuli=selected_stimulus_labels,
    response_range=(-100.0, 100.0),
    response_range_error=False,
    verbose=True,
)

## Model Fitting

In [ ]:
# Edit this block to add, remove, or modify models. The fitting code below does not change.
model_specs = {
    "baseline_random_effects": {
        "formula": "response ~ 1",
        "groups": "fish_neuron_id",
        "vc_formula": {"fish": "0 + C(fish_id)"},
        "notes": "Random-effects-only baseline.",
    },
    "descriptive_stimulus_class": {
        "formula": "response ~ hemifield + stimulus_class",
        "groups": "fish_neuron_id",
        "vc_formula": {"fish": "0 + C(fish_id)"},
        "notes": "Descriptive stimulus-class model.",
    },
    "descriptive_position": {
        "formula": "response ~ hemifield + position_id",
        "groups": "fish_neuron_id",
        "vc_formula": {"fish": "0 + C(fish_id)"},
        "notes": "Position model kept separate to avoid collinearity with stimulus_class.",
    },
    "mechanistic_feature_decomposition": {
        "formula": "response ~ hemifield + motion_level + is_bout_like",
        "groups": "fish_neuron_id",
        "vc_formula": {"fish": "0 + C(fish_id)"},
        "notes": "Non-collinear feature-decomposition model; add other binary flags one at a time.",
    },
}

# Examples to try later:
# "response ~ hemifield + motion_level"
# "response ~ hemifield + motion_level + is_bout_like"
# "response ~ hemifield * motion_level"
# "response ~ hemifield + stimulus_class"
# "response ~ hemifield + position_id"
# Avoid combining position_id with is_full_bout/is_bout_like in the same formula.
# Avoid combining stimulus_class with position_id unless you first check matrix rank.
# Avoid combining motion_level with is_flicker when flicker is the only no-motion class.

In [ ]:
fit_results = lme.fit_lme_models(
    response_table,
    model_specs=model_specs,
    reml=False,
    method="lbfgs",
    maxiter=500,
)
model_results = lme.summarize_lme_model_results(fit_results, response_table)

display(model_results["model_comparison"])
display(model_results["fixed_effects"])
display(model_results["random_effects"])

## Model Outputs

In [ ]:
failed_models = model_results["model_comparison"].query("status == 'failed'")
if not failed_models.empty:
    print("Failed models:")
    display(failed_models[["model_name", "formula", "error"]])

successful_models = model_results["model_comparison"].query("status == 'success'")
print("Successful models:", successful_models["model_name"].tolist())

## Plots

In [ ]:
figures = plott.plot_lme_model_outputs(
    model_results,
    response_table=response_table,
    include_intercept=False,
)

for name, (fig, axes) in figures.items():
    print(name)
    plt.show()